# Validating a model

This notebook validates the model notebook 07 trained: its false-alarm rate on noise, its efficiency on injections,
its sensitive volume and its ROC. It runs offline on simulated coloured noise.

The validator draws data of its own (from `ValidationConfig.seed`, never the training seed), scores the model's
**logits**, and summarises them:

- **FAR** in Hz: background triggers per second of analysed noise, with a Poisson interval;
- **efficiency** against SNR at a chosen FAR, per SNR bin, with a binomial (Clopper-Pearson) interval;
- **sensitive volume** (Gpc³) and **sensitive distance** (Mpc) at each FAR, with three intervals;
- **the exact ROC** and its AUC.

A target FAR the background is too short to resolve is reported as unresolved, never as a number.

In [ ]:
import os
os.environ.setdefault("KERAS_BACKEND", "jax")

from pathlib import Path

import numpy as np

import gravyflow as gf
from gravyflow.src.model.examples import (
    DetectionBatches, Gabbard2017, Gabbard2017Config, RunDirectory, TrainingRun, TrainingSchedule,
    materialise,
)

# Runs live outside the package. Set GRAVYFLOW_RUNS to put them elsewhere.
RUN_DIR = Path(os.environ.get("GRAVYFLOW_RUNS", Path.home() / "gravyflow_runs")) / "gabbard_2017_cpu"

from gravyflow.src.model.examples import ValidationDatasetConfig, validation_dataset_args
from gravyflow.src.validate import ValidationConfig, Validator

## The model

`load_model` reads the run's final model and checks its provenance: the file must hold the weights its record names,
and belong to the run's configuration. The model carries the run's record (`run.json`): the split its weights
trained on and the split that selected them.

In [ ]:
run_dir = RunDirectory(RUN_DIR)
model = run_dir.load_model("model", run_dir.run["config_sha256"])
data = run_dir.run["description"]["config"]
print("sample rate:", data["sample_rate_hertz"], "Hz")

## Validation

4096 noise windows (4096 s of background) and 4096 injections at SNR uniform in 0-15. Results are checkpointed to
`validation.h5`, which records the sha256 of the model and of these settings: running the cell again reuses it, and a
different model or setting is refused rather than given stale scores. The validator reads the model's record: the
split it trained on and the split that selected it (`validate`), and refuses to score that split or data split by
another spec. It scores the `test` split by default.

In [ ]:
dataset_args = validation_dataset_args(
    noise_type=gf.NoiseType.COLORED,
    config=ValidationDatasetConfig(sample_rate_hertz=data["sample_rate_hertz"]),
)
validator = Validator.validate(
    model=model,
    name="gabbard_2017_cpu",
    dataset_args=dataset_args,
    config=ValidationConfig(num_examples=4096, batch_size=256),
    checkpoint_file_path=RUN_DIR / "validation" / "validation.h5",
)

## Metrics

`validator.background()` is the noise scores as a `Background`; `threshold_for_far` gives the score threshold for a
target FAR, with the FAR it achieves and its interval. `efficiency_curves` applies those thresholds to the
injections.

This background is simulated noise, so its windows are independent. On real noise of two or more
detectors the validator measures the background by time slides instead: each detector's data slid
against the others' (`ValidationConfig.num_time_slides`, `time_slide_step_seconds`), inside the held-out
split, tiled into windows whose above-threshold neighbours are one trigger (`cluster_seconds`), with
each slide's livetime counted. Real events (`score_real_events`) are read and conditioned exactly as
that background is. They are real data, read under the noise's split, or beside simulated noise like
this the default split, and the validator checks the model against that split before reading any.

In [ ]:
background = validator.background()
print(f"{background.num_windows} windows, {background.livetime_seconds:.0f} s of background")
for far, (threshold, curve) in validator.efficiency_curves([1e-1, 1e-2, 1e-3, 1e-4]).items():
    if not threshold.resolved:
        print(f"FAR {far:.0e} Hz: unresolved with {background.livetime_seconds:.0f} s of background")
        continue
    achieved = threshold.achieved
    print(f"FAR {far:.0e} Hz: logit threshold {threshold.threshold:.2f}, "
          f"achieved {achieved.rate_hz:.1e} Hz (90% interval {achieved.lower_hz:.1e}-{achieved.upper_hz:.1e})")
    for lo, hi, eff, low, up in zip(curve.bin_edges[:-1], curve.bin_edges[1:], curve.efficiency,
                                    curve.lower, curve.upper):
        if lo >= 6:
            print(f"    SNR {lo:4.0f}-{hi:<3.0f} efficiency {eff:.2f} [{low:.2f}, {up:.2f}]")
for population, roc in validator.roc_curves().items():
    print(f"ROC {population}: AUC {roc['roc_auc']:.4f}")

## Sensitive volume

`sensitive_volume_curve` gives the population-averaged volume in which the model finds a source, at each
target FAR, and the sensitive distance, the radius of a sphere of that volume. It is measured on injections
of its own, drawn as an injection campaign draws them: each source at a physical distance uniform in volume
out to the population's reach (by default the distance at which its loudest source has SNR 4), so V is the
found fraction of the reach's volume. Three 90% intervals come with each volume: the injections' finite
number (exact binomial), the background's finite livetime (the FAR's Poisson interval moves the threshold),
and both together, by a bootstrap over injections and background. The joint interval is the one to quote.

An injection is found whenever its window passes the threshold, by its signal or by its noise: at a high FAR
noise alone finds a share of the whole reach. That noise floor is recorded at each FAR, and the points where
it exceeds a tenth of V are marked (`noise_dominated`).

In [ ]:
curve = validator.sensitive_volume_curve()
print(f"Population uniform in volume to {curve.max_distance_mpc:.0f} Mpc")
# The highest and the lowest FAR the background resolves.
for i in np.flatnonzero(curve.resolved)[[0, -1]]:
    print(f"FAR {curve.far_hz[i]:.1e} Hz: V {curve.volume_gpc3[i]:.3g} Gpc^3 "
          f"(joint 90% interval {curve.joint_lower_gpc3[i]:.3g}-{curve.joint_upper_gpc3[i]:.3g}), "
          f"R {curve.distance_mpc[i]:.0f} Mpc; noise floor {curve.noise_floor_gpc3[i]:.3g} Gpc^3")
print(f"Noise-dominated at {curve.noise_dominated.sum()} of {curve.resolved.sum()} resolved FARs")

## The dashboard

`plot` builds an interactive report: efficiency curves with their intervals, sensitive volume and distance
with their three bands, ROC curves, FAR curves with their Poisson bands, the parameter space, and the
worst-scoring examples.

In [ ]:
dashboard = validator.plot(RUN_DIR / "validation" / "validation.html")
dashboard